#Run Sentiment Analysis on news article

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return None
    return sia.polarity_scores(text)["compound"]

combined_df["sentiment"] = combined_df["content"].apply(get_sentiment)

print("Average sentiment:", combined_df["sentiment"].mean())
print("Done.")

#Splitting the data into categories based on keywords

In [ ]:
import pandas as pd

# -----------------------------
# Categories and keywords
# -----------------------------
categories = {
    "Culture & heritage": ["church", "cathedral", "historic", "heritage", "traditional", "culture", "museum", "monument", "statue", "saint", "art", "chapel", "galleria", "craft", "campanile", "santo", "santa", "fresco", "altar", "altare", "duomo"],
    "Commerce": ["shop", "market", "restaurant", "cafe", "store", "vendor", "commerce", "mercato", "pasticceria", "maschere", "enoteca", "alimentari", "grocery", "souvenir shop", "souvenir", "gift shop"],
    "Tourism": ["tourist", "travel", "sightseeing", "landmark", "visit", "holiday", "turisti", "vacanza", "tours"],
    "Public space life": ["street", "people", "crowd", "sitting", "walking", "public", "social", "square", "campo", "courtyard", "corte", "public benches", "outdoor seating", "community"],
    "Water & mobility": ["canal", "boat", "gondola", "bridge", "vaporetto", "water", "lagoon", "ACTV", "waterbus", "barca", "motorboat", "motoscafo", "pontile", "dock", "molo", "tide", "acqua alta"],
    "Nature & environment": ["tree", "albero", "plants", "piante", "greenery", "garden", "giardino", "orti", "park", "parco", "sky", "sunset", "sunrise", "environmental", "ecology", "ecologia", "vegetation", "flowers", "climate", "environment", "weather", "clouds", "fog"],
    "Events": ["festival", "event", "carnival", "performance", "parade", "concert", "Carnevale di Venezia", "celebration", "feast", "festa", "fireworks", "fuochi d’artificio"],
    "Architecture": ["building", "edificio", "facade", "facciata", "arch", "arcade", "arco", "column", "colonna", "capital", "window", "finestra", "balcony", "balcone", "bridge", "ponte", "dome", "cupola", "tower", "torre", "bell tower", "campanile", "roof", "tetto", "chimney", "comignolo"]
}

# -----------------------------
# Categorize each row
# -----------------------------
def categorize(row):
    title = str(row.get("title", "")).lower()
    link = str(row.get("link", "")).lower()  # optional
    matched_categories = []

    for cat, keywords in categories.items():
        for kw in keywords:
            if kw.lower() in title or kw.lower() in link:
                matched_categories.append(cat)
                break  # stop at first keyword match per category

    if not matched_categories:
        return "Uncategorized"
    else:
        return ", ".join(matched_categories)

# Apply to DataFrame
df['category'] = df.apply(categorize, axis=1)

# -----------------------------
# Save the categorized CSV
# -----------------------------
df.to_csv("/content/drive/MyDrive/Venice_RC15/Datasets/News/combined_news_categorized.csv", index=False)
print("Categorized CSV saved.")


In [ ]:
#Checking the distribution of categories

In [ ]:
import pandas as pd

# -----------------------------
# Count number of data points per category
# -----------------------------
# Split multiple categories into separate rows for counting
df_categories = df['category'].str.split(',', expand=True).stack().str.strip()
category_counts = df_categories.value_counts()

print("Number of data points in each category:")
print(category_counts)

#Visualizing this data

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Load your categorized CSV
df = pd.read_csv("/content/drive/MyDrive/Venice_RC15/Datasets/News/combined_news_categorized.csv")

# -----------------------------
# Prepare source-target counts
# -----------------------------
# Some articles may have multiple categories, split them
df_expanded = df.copy()
df_expanded['category'] = df_expanded['category'].str.split(',')
df_expanded = df_expanded.explode('category')
df_expanded['category'] = df_expanded['category'].str.strip()

# Count the number of occurrences for each source_file -> category
flow_counts = df_expanded.groupby(['source_file', 'category']).size().reset_index(name='count')

# -----------------------------
# Prepare Sankey labels and indexes
# -----------------------------
all_nodes = list(flow_counts['source_file'].unique()) + list(flow_counts['category'].unique())
node_indices = {name: i for i, name in enumerate(all_nodes)}

# Map source and target to node indices
flow_counts['source_idx'] = flow_counts['source_file'].map(node_indices)
flow_counts['target_idx'] = flow_counts['category'].map(node_indices)

# -----------------------------
# Create Sankey diagram
# -----------------------------
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=all_nodes,
        color="blue"
    ),
    link=dict(
        source=flow_counts['source_idx'],
        target=flow_counts['target_idx'],
        value=flow_counts['count']
    )
)])

fig.update_layout(title_text="Flow of News Sources to Categories", font_size=12)
fig.show()
